# A hidden layer learns XOR

XOR is the smallest useful example of representation learning: no single straight boundary separates the classes, but a small hidden layer can combine several boundaries.

## 1. Using a standard library
Let scikit-learn manage initialization, forward passes, gradients, and optimization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

rng = np.random.default_rng(3)
corners = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
labels = np.array([0, 1, 1, 0])
X = np.repeat(corners, 60, axis=0) + rng.normal(0, .09, (240, 2))
y = np.repeat(labels, 60)

net = MLPClassifier(hidden_layer_sizes=(4,), activation='tanh', learning_rate_init=.03,
                    max_iter=3000, random_state=3).fit(X, y)
print('Probabilities at the four corners:', net.predict_proba(corners)[:, 1].round(3))

xx, yy = np.meshgrid(np.linspace(-.3, 1.3, 220), np.linspace(-.3, 1.3, 220))
prob = net.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].contourf(xx, yy, prob, levels=np.linspace(0, 1, 11), cmap='RdBu', alpha=.65)
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='white', s=18)
axes[0].set_title('Learned probability surface')
axes[1].plot(net.loss_curve_); axes[1].set(title='Training loss', xlabel='iteration', ylabel='loss')
plt.tight_layout();

## 2. Build the network from scratch
Write the hidden activations, output probabilities, and backpropagation updates directly with NumPy.

In [ ]:
target = y[:, None].astype(float)
rng_scratch = np.random.default_rng(3)
W1 = rng_scratch.normal(0, .5, (2, 4)); b1 = np.zeros((1, 4))
W2 = rng_scratch.normal(0, .5, (4, 1)); b2 = np.zeros((1, 1))

for _ in range(3000):
    hidden = np.tanh(X @ W1 + b1)
    probability = 1 / (1 + np.exp(-(hidden @ W2 + b2)))
    d2 = (probability - target) / len(X)
    gW2 = hidden.T @ d2; gb2 = d2.sum(axis=0, keepdims=True)
    d1 = (d2 @ W2.T) * (1 - hidden ** 2)
    gW1 = X.T @ d1; gb1 = d1.sum(axis=0, keepdims=True)
    W1 -= .3 * gW1; b1 -= .3 * gb1
    W2 -= .3 * gW2; b2 -= .3 * gb2

corner_hidden = np.tanh(corners @ W1 + b1)
corner_probability = 1 / (1 + np.exp(-(corner_hidden @ W2 + b2)))
print('From-scratch corner probabilities:', corner_probability.ravel().round(3))

## What to notice
The hidden units make a nonlinear decision surface possible. This example has abundant stable signal. In financial prediction, greater capacity is helpful only if similarly stable structure exists beyond the training sample.